# Remove common words like "the", "and", "is" that appear everywhere and carry no meaning for your concept analysis.
What Chai says

Start with a short stopword list and expand carefully

Never remove "no", "not", "never" — they reverse meaning

Removed stopwords are gone permanently — be cautious

#### 1. Imports

I worked with [this](https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/) documentation. 

In [28]:
import pandas as pd
import ast
import nltk

nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### 2. Load Dataset

In [29]:
df_h1 = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/07_h1_normalized.csv")
df_h1["tokens"] = df_h1["tokens"].apply(ast.literal_eval)

#### 3. Checking the default stopwords list

In [30]:
stop_words = set(stopwords.words("english"))
print(stop_words)
print(f"Total stopwords: {len(stop_words)}")

{"she'd", "we'd", 't', 'needn', "aren't", 'they', 'before', 'don', 'ma', 'any', 'as', 'been', 'out', 'through', 'where', 'only', 'how', "i'm", "they'd", 've', 'couldn', 'both', 'further', 'itself', 'theirs', 'no', 'them', 'yourselves', 'once', 'then', "should've", 'shouldn', 'being', 'into', 'mightn', 'while', 'again', 'over', 'whom', 'his', 'should', "mightn't", 'a', 'under', 'but', 'ain', 'of', "you'll", 'nor', 'after', 'hasn', 'about', 'some', 'which', "we're", "weren't", 'will', 'do', 'haven', 'him', 'not', 'shan', 'she', 'if', "mustn't", 'during', "that'll", 'who', 'o', 'am', 'he', "hasn't", 'to', 'by', 'i', 'isn', 'mustn', "he's", 'at', 'few', 'against', 'below', 'doesn', 'it', "it'll", 'does', 'here', 'didn', 'wasn', 'm', 'for', "wouldn't", "she's", 'you', 'himself', 'off', 'because', 'too', 'have', 'up', "he'd", 'those', 'is', 'with', 'these', 'my', 'weren', 'other', "it's", 'that', "they'll", 'when', "he'll", 'her', 'until', 's', 'down', "needn't", "you've", 'me', 'has', 'so',

#### 4. Removing Negations from the stopwords list
Referencing Chai. 

In [31]:
negations = {"no", "not", "nor", "never", "neither", "nothing", "nowhere", "nobody"}
boilerplate = {"idea-oriented", "heresy", "heresies", "devote", "journal", "publication"}
stop_words = stop_words | boilerplate
stop_words = stop_words - negations

#### 5. Writing the Function

In [32]:
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

##### Tokens before stopword removal

In [33]:
print(f"H1 total tokens before stopword removal: {df_h1['tokens'].apply(len).sum()}")

H1 total tokens before stopword removal: 147965


#### 6. Apply the function

In [34]:
df_h1["tokens"] = df_h1["tokens"].apply(remove_stopwords)

#### 7. Checking if it worked

In [35]:
print(df_h1["tokens"].iloc[0])
print(f"H1 total tokens after stopword removal: {df_h1['tokens'].apply(len).sum()}")

['devoted', 'examination', 'art', 'politics']
H1 total tokens after stopword removal: 80180


##### The following cell was created with Claude. Please find the Prompt for this here: 

/Users/sophiehamann/master-thesis-code/notebooks/AI_prompts/h1_09_stopword_check_prompt.txt

In [36]:
# --- Stopword Removal Audit ---

# Rebuild a flat list of all original tokens (reload before removal)
df_check = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/07_h1_normalized.csv")
df_check["tokens"] = df_check["tokens"].apply(ast.literal_eval)

all_original = [token for tokens in df_check["tokens"] for token in tokens]
all_kept     = [token for tokens in df_h1["tokens"]    for token in tokens]

original_vocab = set(all_original)
kept_vocab     = set(all_kept)
removed_vocab  = original_vocab - kept_vocab

# 1. Which unique words were removed?
print(f"Unique words removed: {len(removed_vocab)}")
print(sorted(removed_vocab), "\n")

# 2. How often did each removed word appear? (top 30 by frequency)
from collections import Counter
original_counts = Counter(all_original)
removed_counts  = {w: original_counts[w] for w in removed_vocab}
top_removed     = sorted(removed_counts.items(), key=lambda x: x[1], reverse=True)[:30]
print("Top 30 removed words by frequency:")
for word, count in top_removed:
    print(f"  {word:20s} {count:>6}")

# 3. Spot-check: are any negations in the removed set? (should be empty)
negations = {"no", "not", "nor", "never", "neither", "nothing", "nowhere", "nobody"}
removed_negations = negations & removed_vocab
print(f"\nNegations accidentally removed: {removed_negations or 'none — all good'}")

# 4. Reduction summary
print(f"\nTokens before : 147965")
print(f"Tokens after  : 80393")
print(f"Reduction     : {(1 - 80393/147965)*100:.1f}%")

Unique words removed: 141
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'd', 'devote', 'did', 'do', 'does', 'doing', 'don', 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'has', 'have', 'haven', 'having', 'he', 'her', 'here', 'heresies', 'heresy', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', 'idea-oriented', 'if', 'in', 'into', 'is', 'it', 'its', 'itself', 'journal', 'just', 'll', 'm', 'ma', 'me', 'more', 'most', 'my', 'myself', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 'publication', 're', 's', 'same', 'she', 'should', 'so', 'some', 'such', 't', 'than', 'that', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'we',

I went from 147.965 tokens to 80393 tokens which is a 45.7% reduction of tokens. This matches what Chai says (40-50% reduction).
I NEED TO QUOTE THIS CORRECTLY.

#### 8. Saving

In [37]:
df_h1.to_csv("/Users/sophiehamann/master-thesis-code/data/processed/08_h1_stopwords.csv", index=False)